<a href="https://colab.research.google.com/github/mrunmayee3108/NeuroSolve/blob/main/qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
!pip install trl
from trl import SFTTrainer

In [ ]:
codex_df = pd.read_csv("codex_clean.csv")
mqa_df = pd.read_csv("unified_mqa_train.csv")

In [ ]:
master_train_df = pd.concat([codex_df, mqa_df], ignore_index=True)
master_train_df = master_train_df.sample(frac=1, random_state=42).reset_index(drop=True)
dataset = Dataset.from_pandas(master_train_df)

In [ ]:
model_id = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
tokenizer.padding_side = "right"

In [ ]:
def format_prompt(row):
    """The strict format we force the AI to learn."""
    clean_code = str(row['code']).replace("```python", "").replace("```", "").strip()
    prompt = f"### Instruction: Write Python code to solve the math problem. Store the answer in 'result'.\n"
    prompt += f"### Question:\n{row['question']}\n"
    prompt += f"### Code:\n```python\n{clean_code}\n```"
    return {"text": prompt + tokenizer.eos_token}
formatted_dataset = dataset.map(format_prompt)

In [ ]:
print("LOADING MODEL IN 4-BIT")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)

# Ensure config.rope_scaling has the necessary keys for 'longrope' type
if not hasattr(config, 'rope_scaling') or config.rope_scaling is None:
    config.rope_scaling = {'type': 'longrope', 'short_factor': 1.0, 'long_factor': 1.0}
elif isinstance(config.rope_scaling, dict):
    if 'type' not in config.rope_scaling:
        config.rope_scaling['type'] = 'longrope'
    if config.rope_scaling['type'] == 'longrope':
        if 'short_factor' not in config.rope_scaling:
            config.rope_scaling['short_factor'] = 1.0  # Default value
        if 'long_factor' not in config.rope_scaling:
            config.rope_scaling['long_factor'] = 1.0   # Default value

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config, # Pass the modified config
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager"
)
model = prepare_model_for_kbit_training(model)

In [ ]:
print("APPLYING QLoRA")
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

In [ ]:
print("TRAINING")
training_args = TrainingArguments(
    output_dir="./phi3-math-agent",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=50,
    bf16=True,
    optim="paged_adamw_8bit"
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    args=training_args
)
trainer.train()

In [ ]:
print("SAVING ---")
trainer.model.save_pretrained("phi3-neuro-symbolic-adapter")
tokenizer.save_pretrained("phi3-neuro-symbolic-adapter")
print("TRAINING COMPLETE!")

In [ ]:
!zip -r phi3-neuro-symbolic-adapter.zip phi3-neuro-symbolic-adapter

In [ ]:
from google.colab import files
files.download("phi3-neuro-symbolic-adapter.zip")